In [84]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from tqdm import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [85]:
root_path = "/home/stefan/ioai-prep/kits/pre-iaio/meaning-of-life"
seed = 42

# Data

In [86]:
df = pd.read_csv(f"{root_path}/train_data.csv")
df.head()

,subtaskID,datapointID,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10
0,2,1,18.845863,15.432435,-4.888494,-11.196174,1.408862,-17.684394,3.231677,-1.476026,-4.660365,-15.947031
1,2,2,-4.746868,-5.971201,4.346016,-6.677494,1.910919,7.616225,-0.325792,-8.837899,-10.356530,-4.331975
2,2,3,2.908971,-19.955318,-11.212437,9.294100,-1.321366,-5.237313,-13.585740,-10.797363,-20.217986,-1.012807
3,2,4,-12.374266,1.094085,13.286408,3.131845,-6.065034,4.559042,-4.590903,-6.946004,-11.543627,-17.518288
4,2,5,3.261330,-12.511136,9.240270,-1.849021,-5.227230,10.490092,-7.043437,-14.084613,-15.566292,6.060100


# Subtask 1

In [87]:
# is 51 prime?
# used a few quantum computers for calculations
subtask1 = 0

# Subtask 2: pca method

In [88]:
X = df[[f"X{i}" for i in range(1, 11)]]

In [89]:
pca = PCA(n_components=3)
X_reduced = pca.fit_transform(X)
X_reconstructed = pca.inverse_transform(X_reduced)

In [90]:
errors = np.linalg.norm(X - X_reconstructed, axis=1)

threshold = np.mean(errors) * 0.5
good_indices = np.where(errors <= threshold)[0]
len(good_indices)

16

# Subtask 2 - ransac-like method

In [91]:
X_np = X.values
N, D = X_np.shape
n_inliers = 20
n_dims_subspace = 3

iterations = 200000

best_error = float("inf")
best_indices = None

In [ ]:
for i in tqdm(range(iterations)):
    # sample 4 points
    sample_idx = np.random.choice(N, n_dims_subspace + 1, replace=False)
    sample_pts = X_np[sample_idx]

    # create subspace
    mean = np.mean(sample_pts, axis=0)
    centered_sample = sample_pts - mean
    _, _, Vt = np.linalg.svd(centered_sample, full_matrices=False)
    basis = Vt[:n_dims_subspace]

    # distance as error
    X_centered = X_np - mean
    norms_X = np.sum(X_centered**2, axis=1)
    norms_proj = np.sum((X_centered @ basis.T) ** 2, axis=1)
    errors = np.sqrt(np.maximum(0, norms_X - norms_proj))

    # partial sort for speed
    top_20_idx = np.argpartition(errors, n_inliers - 1)[:n_inliers]
    total_error = np.sum(errors[top_20_idx])

    # refinement step (refit)
    if total_error < best_error:
        refine_pts = X_np[top_20_idx]
        refine_mean = np.mean(refine_pts, axis=0)
        refine_centered = refine_pts - refine_mean

        _, _, refine_Vt = np.linalg.svd(refine_centered, full_matrices=False)
        refine_basis = refine_Vt[:n_dims_subspace]

        refine_X_centered = X_np - refine_mean
        refine_norms_X = np.sum(refine_X_centered**2, axis=1)
        refine_norms_proj = np.sum((refine_X_centered @ refine_basis.T) ** 2, axis=1)
        refine_errors = np.sqrt(np.maximum(0, refine_norms_X - refine_norms_proj))

        refine_top_20_idx = np.argpartition(refine_errors, n_inliers - 1)[:n_inliers]
        refine_total_error = np.sum(refine_errors[refine_top_20_idx])

        if refine_total_error < best_error:
            best_error = refine_total_error
            best_indices = refine_top_20_idx.copy()
            print(f"Iteration {i} - New best error: {best_error:.4f}")

  0%|          | 0/200000 [00:00<?, ?it/s]

  1%|          | 1864/200000 [00:00<00:21, 9427.09it/s]

Iteration 0 - New best error: 203.8884
Iteration 5 - New best error: 169.9613
Iteration 35 - New best error: 154.7838
Iteration 75 - New best error: 139.6951


  6%|▌         | 11492/200000 [00:01<00:19, 9567.66it/s]

Iteration 9979 - New best error: 133.7358


...

 34%|███▎      | 67469/200000 [00:07<00:13, 9792.93it/s]

Iteration 65604 - New best error: 89.0606


 36%|███▌      | 71365/200000 [00:07<00:13, 9346.75it/s]

Iteration 69899 - New best error: 46.6672


100%|██████████| 200000/200000 [00:21<00:00, 9452.61it/s]


In [95]:
good_indices

array([  3,  78, 114, 124, 160, 239, 273, 295, 305, 310, 340, 345, 357,
       377, 381, 414])

In [94]:
best_indices

array([399, 348, 354,  78, 265,   1, 357, 124, 239, 414, 273, 310, 276,
       381, 267, 387, 169,  34, 134, 305])

In [96]:
subtask2 = np.zeros((X.shape[0],), dtype=int)
subtask2[best_indices] = 1

# Submission

In [97]:
def build_subtask(ans, sid):
    return pd.DataFrame({"subtaskID": sid, "datapointID": df["datapointID"] if sid == 2 else [1], "answer": ans})


subtasks = [
    (subtask1, 1),
    (subtask2, 2),
]

submission_df = pd.concat([build_subtask(ans, sid) for ans, sid in subtasks])

In [98]:
submission_df.head()

,subtaskID,datapointID,answer
0,1,1,0
0,2,1,0
1,2,2,1
2,2,3,0
3,2,4,0


In [99]:
submission_df.to_csv(f"{root_path}/submission.csv", index=False)